In [ ]:
! pip install pandas openpyxl
import pandas as pd
import os
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

# Extract 36 Skeleton Points

In [ ]:
# -- Basic helper functions --
def col_letter(idx: int) -> str:
    """
    1-based column index → Excel-style column letters
    Convert 1-based column index to Excel-style letters
    """
    letters = ""
    while idx:
        idx, rem = divmod(idx - 1, 26)
        letters = chr(65 + rem) + letters
    return letters

def group_to_range(n: int) -> str:
    """
    Group number → 3-column range string
    Map group number to a 3-column wide range string
    """
    start = n * 3 + 1        # Start column (1-based)
    end   = start + 2        # End column
    return f"{col_letter(start)}:{col_letter(end)}"

# -- Your index list --
nums = [0, 1, 2, 3, 4, 5, 6, 7, 15, 16, 17, 18, 27, 29, 33, 37, 39, 44, 49, 52, 73, 76, 79, 82, 95, 99, 103, 107, 111, 115, 116, 120, 124, 128, 132, 136]
print(len(nums))
ranges = [group_to_range(n) for n in nums]
usecols_str = ",".join(ranges)
print(usecols_str)


In [ ]:
# Specify columns to read
usecols_str = "A:C,D:F,G:I,J:L,M:O,P:R,S:U,V:X,AT:AV,AW:AY,AZ:BB,BC:BE,CD:CF,CJ:CL,CV:CX,DH:DJ,DN:DP,EC:EE,ER:ET,FA:FC,HL:HN,HU:HW,ID:IF,IM:IO,JZ:KB,KL:KN,KX:KZ,LJ:LL,LV:LX,MH:MJ,MK:MM,MW:MY,NI:NK,NU:NW,OG:OI,OS:OU"

def process_sample(sampleID):
    skeletonPath = os.path.join(data_path, sampleID, sampleID + '_openpose.xlsx')
    if not os.path.exists(skeletonPath):
        print(f"[Error] Skeleton data not found for {sampleID}")
        return
    try:
        df = pd.read_excel(skeletonPath, engine='openpyxl', dtype=object, header=None, usecols=usecols_str)
        skeletonPath_new = os.path.join(output_path, sampleID + '_36.csv')
        df.to_csv(skeletonPath_new, index=False)
        print(f"[OK] {sampleID} processed successfully")
    except Exception as e:
        print(f"[Fail] Error processing {sampleID}: {e}")


In [ ]:
# Train RGB clips
data_path = "../MiGA/imigue_data_phase1/datasets/imigue_skeleton_train"
output_path = "../MiGA/skeleton/train"
os.makedirs(output_path, exist_ok=True)
sample_list = os.listdir(data_path)
with ProcessPoolExecutor(max_workers=16) as executor:
    executor.map(process_sample, sample_list)

In [ ]:
# Val RGB clips
data_path = "../MiGA/imigue_data_phase1/datasets/imigue_skeleton_validate"
output_path = "../MiGA/skeleton/val"
os.makedirs(output_path, exist_ok=True)
sample_list = os.listdir(data_path)
with ProcessPoolExecutor(max_workers=16) as executor:
    executor.map(process_sample, sample_list)

In [ ]:
# Test RGB clips
data_path = "../MiGA/imigue_data_phase2/imigue_skeleton_test"
output_path = "../MiGA/skeleton/test"
os.makedirs(output_path, exist_ok=True)
sample_list = os.listdir(data_path)
with ProcessPoolExecutor(max_workers=16) as executor:
    executor.map(process_sample, sample_list)

# Write to pkl file as required

In [ ]:
import os
import csv
import cv2
import numpy as np
from tqdm import tqdm
import pickle

In [ ]:
np.__version__  # < 2.0.0

In [ ]:
def read_csv_to_list(file_path):
    """
    Read a CSV file and return a 2D list (list of lists)
    :param file_path: Path to the CSV file
    :return: List of rows, each row is a sub-list
    """
    data_list = []
    with open(file_path, mode='r', encoding='utf-8', newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            # Convert each element in row from string to int
            int_row = [int(value) for value in row]
            data_list.append(int_row)
    return data_list

def read_all_labels(root_dir):
    """
    Traverse all subfolders under the root directory, read the corresponding xxx_label.csv file,
    and store the results in a dictionary to return.
    :param root_dir: Path to the root directory
    :return: { folder_name: data_list, ... }
    """
    all_data = {}
    # List all entries in root_dir
    for entry in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, entry)
        # Only process directories
        if os.path.isdir(folder_path):
            # Construct expected CSV filename
            csv_name = f"{entry}_label.csv"
            csv_path = os.path.join(folder_path, csv_name)
            # Check if file exists
            if os.path.isfile(csv_path):
                # Read and save to dictionary
                all_data[entry] = read_csv_to_list(csv_path)
            else:
                print(f"Warning: File not found {csv_path}")
    return all_data

In [ ]:
# Dataset root directory
dataset_root = "../MiGA/imigue_data_phase1/datasets/imigue_skeleton_train"
skeleton_data = "../MiGA/skeleton/train"
# Read all label data
label_data_dict = read_all_labels(dataset_root)
OUTPUT = {
    "split":{"train":[]},
    "annotations":[]
}

for folder, data in label_data_dict.items():
    print(f"--- {folder} ---")

    frames = {
        "keypoint":[],
        "keypoint_score":[]
    }

    ske = skeleton_data+f"/{folder}_36.csv"
    with open(ske, mode='r', encoding='utf-8', newline='') as csvfile:
        reader = csv.reader(csvfile)
        for idx, row in enumerate(tqdm(reader)):
            if idx == 0:
                continue
            # Convert each element in row from string to float
            float_row = [float(value) for value in row]
            coords = []
            conf = []

            for i in range(0, len(float_row), 3):
                x, y, c = float_row[i], float_row[i+1], float_row[i+2]
                coords.append([x, y])
                conf.append(c)
            frames["keypoint"].append(coords)
            frames["keypoint_score"].append(conf)

    L = set()
    for ids, row in enumerate(data):
        if len(row)!=3:
            print("err! row!=3")
            break
            
        # Create new annotations in the loop
        annotations = {
            'frame_dir': f"{folder}_{ids}",
            'label': 31 if row[0] == 99 else row[0] - 1,
            'total_frames': row[2] - row[1] + 1,
            'img_shape':    (720, 1280),
            'original_shape': (720, 1280),
            'keypoint':     np.array([frames["keypoint"][row[1]:row[2]+1]], dtype=np.float32),
            'keypoint_score': np.array([frames["keypoint_score"][row[1]:row[2]+1]], dtype=np.float32)
        }

        OUTPUT["split"]["train"].append(annotations["frame_dir"])
        OUTPUT["annotations"].append(annotations)
        L.add(annotations['label'])
    print(L)

with open('../Skeleton/iMiGUE_train.pkl', 'wb') as file:
    pickle.dump(OUTPUT, file)
print("Done!")

In [ ]:
# Dataset root directory
dataset_root = "../MiGA/imigue_data_phase1/datasets/imigue_skeleton_validate"
skeleton_data = "../MiGA/skeleton/val"
# Read all label data
label_data_dict = read_all_labels(dataset_root)
OUTPUT = {
    "split":{"val":[]},
    "annotations":[]
}

for folder, data in label_data_dict.items():
    print(f"--- {folder} ---")

    frames = {
        "keypoint":[],
        "keypoint_score":[]
    }

    ske = skeleton_data+f"/{folder}_36.csv"
    with open(ske, mode='r', encoding='utf-8', newline='') as csvfile:
        reader = csv.reader(csvfile)
        for idx, row in enumerate(tqdm(reader)):
            if idx == 0:
                continue
            # Convert each element in row from string to float
            float_row = [float(value) for value in row]
            coords = []
            conf = []

            for i in range(0, len(float_row), 3):
                x, y, c = float_row[i], float_row[i+1], float_row[i+2]
                coords.append([x, y])
                conf.append(c)
            frames["keypoint"].append(coords)
            frames["keypoint_score"].append(conf)

    L = set()
    for ids, row in enumerate(data):
        if len(row)!=3:
            print("err! row!=3")
            break
            
        # Create new annotations in the loop
        annotations = {
            'frame_dir': f"{folder}_{ids}",
            'label': 31 if row[0] == 99 else row[0] - 1,
            'total_frames': row[2] - row[1] + 1,
            'img_shape':    (720, 1280),
            'original_shape': (720, 1280),
            'keypoint':     np.array([frames["keypoint"][row[1]:row[2]+1]], dtype=np.float32),
            'keypoint_score': np.array([frames["keypoint_score"][row[1]:row[2]+1]], dtype=np.float32)
        }

        OUTPUT["split"]["val"].append(annotations["frame_dir"])
        OUTPUT["annotations"].append(annotations)
        L.add(annotations['label'])
    print(L)

with open('../Skeleton/iMiGUE_val.pkl', 'wb') as file:
    pickle.dump(OUTPUT, file)
print("Done!")

In [ ]:
# Dataset root directory
dataset_root = "../MiGA/imigue_data_phase2/imigue_skeleton_test"
skeleton_data = "../MiGA/skeleton/test"
# Read all label data
label_data_dict = read_all_labels(dataset_root)
OUTPUT = {
    "split":{"test":[]},
    "annotations":[]
}

for folder, data in label_data_dict.items():
    print(f"--- {folder} ---")

    frames = {
        "keypoint":[],
        "keypoint_score":[]
    }

    ske = skeleton_data+f"/{folder}_36.csv"
    with open(ske, mode='r', encoding='utf-8', newline='') as csvfile:
        reader = csv.reader(csvfile)
        for idx, row in enumerate(tqdm(reader)):
            if idx == 0:
                continue
            # Convert each element in row from string to float
            float_row = [float(value) for value in row]
            coords = []
            conf = []

            for i in range(0, len(float_row), 3):
                x, y, c = float_row[i], float_row[i+1], float_row[i+2]
                coords.append([x, y])
                conf.append(c)
            frames["keypoint"].append(coords)
            frames["keypoint_score"].append(conf)

    L = set()
    for ids, row in enumerate(data):
        if len(row)!=3:
            print("err! row!=3")
            break
            
        # Create new annotations in the loop
        annotations = {
            'frame_dir': f"{folder}_{ids}",
            'label': 31 if row[0] == 99 else row[0] - 1,
            'total_frames': row[2] - row[1] + 1,
            'img_shape':    (720, 1280),
            'original_shape': (720, 1280),
            'keypoint':     np.array([frames["keypoint"][row[1]:row[2]+1]], dtype=np.float32),
            'keypoint_score': np.array([frames["keypoint_score"][row[1]:row[2]+1]], dtype=np.float32)
        }

        OUTPUT["split"]["test"].append(annotations["frame_dir"])
        OUTPUT["annotations"].append(annotations)
        L.add(annotations['label'])
    print(L)

with open('../Skeleton/iMiGUE_test.pkl', 'wb') as file:
    pickle.dump(OUTPUT, file)
print("Done!")

In [ ]:
# Merge pkl files
import pickle
import numpy
numpy.__version__

with open('../Skeleton/iMiGUE_train.pkl', 'rb') as f:
    iMiGUE_train = pickle.load(f)
with open('../SkeletoniMiGUE_val.pkl', 'rb') as f:
    iMiGUE_val = pickle.load(f)
with open('../SkeletoniMiGUE_test.pkl', 'rb') as f:
    iMiGUE_test = pickle.load(f)

OUTPUT = {
    'split': {
        'train': iMiGUE_train['split']['train'],
        'val': iMiGUE_val['split']['val'],
        'test': iMiGUE_test['split']['test']
    }, 
    'annotations': iMiGUE_train['annotations']+iMiGUE_val['annotations']+iMiGUE_test['annotations']
}
with open('../Skeleton/iMiGUE_36.pkl', 'wb') as f:
    pickle.dump(OUTPUT, f)